In [1]:
import pandas as pd

# 2024 populations (countryeconomy.com / Destatis, 2024)
pop = {
    '미국': 340.1, '독일': 83.6, '프랑스': 68.9, '이탈리아': 58.9, '스페인': 49.1, '영국': 68.3,
    '한국': 51.75, '일본': 123.9, '중국': 1408.3  # 단위: 백만 명
}
pop['EU5'] = pop['독일'] + pop['프랑스'] + pop['이탈리아'] + pop['스페인'] + pop['영국']

# --- GD 환자 풀 ---
# 유병률 가정: 기본 0.5% (글로벌 통용), 중국 0.53% (Wang 2021), 한국 0.20% (치료된 갑상선기능항진증 0.276%의 GD 비중 ~70% 가정)
# 발생률: 덴마크 26.8/10만 (Klit 2024), 스웨덴 예테보리 21.4/10만, 한국 치료 기능항진증 55/10만의 GD 비중 ~60% -> 33/10만
gd_prev = {'미국': 0.005, 'EU5': 0.005, '한국': 0.002, '일본': 0.005, '중국': 0.0053}
gd_inc = {'미국': 26.8, 'EU5': 26.8, '한국': 33.0, '일본': 26.8, '중국': 26.8}  # /10만/년

rows = []
for r in ['미국', 'EU5', '한국', '일본', '중국']:
    p = pop[r]
    rows.append({
        '지역': r, '인구(2024, 백만)': round(p, 1),
        'GD 유병률 가정(%)': gd_prev[r]*100,
        'GD 유병 환자수(백만)': round(p*gd_prev[r], 2),
        'GD 연간 신규 발생률(/10만)': gd_inc[r],
        'GD 연간 신규 환자수(천 명)': round(p*gd_inc[r]/100, 1),
    })
gd_pool = pd.DataFrame(rows)
print('=== Graves Disease 환자 풀 ===')
print(gd_pool.to_string(index=False))

# --- TED 환자 풀 ---
# 유병률 시나리오: 저(일본 strict 24.65/10만) / 고(미국 Olmsted 65/10만)
# 발생률: 5.0 (덴마크/미국 Olmsted) ~ 11.1 (일본 broad) /10만/년
ted_prev_low, ted_prev_high = 24.65, 65.0   # /10만
ted_inc_low, ted_inc_high = 5.0, 11.1       # /10만/년

rows = []
for r in ['미국', 'EU5', '한국', '일본', '중국']:
    p = pop[r]
    rows.append({
        '지역': r, '인구(2024, 백만)': round(p, 1),
        'TED 유병 환자수-저(천 명)': round(p*ted_prev_low/100, 1),
        'TED 유병 환자수-고(천 명)': round(p*ted_prev_high/100, 1),
        'TED 연간 신규-저(천 명)': round(p*ted_inc_low/100, 1),
        'TED 연간 신규-고(천 명)': round(p*ted_inc_high/100, 1),
    })
ted_pool = pd.DataFrame(rows)
print('\n=== Thyroid Eye Disease 환자 풀 ===')
print(ted_pool.to_string(index=False))

# GD 기반 TED 추정 (Chin 2020 메타: GD 환자의 40%, 아시아 44-45%, 북미 27%, 유럽 38%)
print('\n=== GD 기반 TED 환자수 (유병 GD x TED 동반비율, 경증 포함) ===')
for r in ['미국', 'EU5', '한국', '일본', '중국']:
    gd = pop[r]*gd_prev[r]
    share = {'미국': 0.27, 'EU5': 0.38, '한국': 0.44, '일본': 0.44, '중국': 0.44}[r]
    print(f"{r}: GD {gd:.2f}M x {int(share*100)}% = {gd*share:.2f}M")

=== Graves Disease 환자 풀 ===
 지역  인구(2024, 백만)  GD 유병률 가정(%)  GD 유병 환자수(백만)  GD 연간 신규 발생률(/10만)  GD 연간 신규 환자수(천 명)
 미국         340.1          0.50           1.70                26.8               91.1
EU5         328.8          0.50           1.64                26.8               88.1
 한국          51.8          0.20           0.10                33.0               17.1
 일본         123.9          0.50           0.62                26.8               33.2
 중국        1408.3          0.53           7.46                26.8              377.4

=== Thyroid Eye Disease 환자 풀 ===
 지역  인구(2024, 백만)  TED 유병 환자수-저(천 명)  TED 유병 환자수-고(천 명)  TED 연간 신규-저(천 명)  TED 연간 신규-고(천 명)
 미국         340.1               83.8              221.1              17.0              37.8
EU5         328.8               81.0              213.7              16.4              36.5
 한국          51.8               12.8               33.6               2.6               5.7
 일본         123.9               30.5               80.

In [2]:
import subprocess
r = subprocess.run(['python3', '/workspace/build_excel.py'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.stderr else 'OK')

saved: /workspace/Graves_TED_landscape.xlsx

OK
